# 03c — Same-day visits to Košice from abroad, direct trains

Can a resident of a place abroad **go to Košice and back on the same day** by a **direct** ZSR train? Per station and
per LAU of Czechia, Hungary, Austria, Poland and Ukraine: the number of days in the country run's window on which such
a visit is possible. Options from 03a (`foreign/direct_legs.parquet`); rule and engine of 02c
(`nb02_connections.daytrip_days`).

| § | step | output (`output/kosice/foreign/`) |
|---|---|---|
| 1 | options: inbound (home → Košice) and outbound (Košice → home), per country run | — |
| 2 | day-trip days per station and per LAU | `daytrip_direct_station.csv`, `daytrip_direct_lau.csv` |
| 3 | coverage per country with its denominators | `daytrip_direct_coverage.csv` |
| 4 | dwell sensitivity (VAL-3) | `daytrip_direct_sensitivity.csv` |
| 5 | against the legacy run | — |

**Rule** (register PIP-16): on one date, an inbound option **arriving by** `daytrip_max_arrival`, at least
`daytrip_min_dwell_min` in Košice, and an outbound option **leaving at or after** `daytrip_min_departure` and **back
by** `daytrip_max_return`. Service-day clock, no wrap: an overnight arrival (after 24:00) or a return after 23:59 is
not a same-day visit. Each country run on its own window (03a); `daytrip_share` = days / window days.

In [1]:
import os, sys

# Run from the project root so every relative path resolves the same from notebooks/ or the root.
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    os.chdir(os.path.dirname(_here))
sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

Working directory: E:\notes\api_zssk\project


In [2]:
import pandas as pd
import traintranv2  # noqa: F401  (PROJ data fix)
from traintranv2.config import default_config
from traintranv2 import nb02_connections as C
from traintranv2 import nb03_foreign as F
from traintranv2.nb01a_feeds import write_meta
from traintranv2.nb01b_stations import load_lau_subset

KIND = 'direct'
cfg = default_config()
OUT = os.path.join(cfg.out_dir, 'foreign')
tags = pd.read_csv(os.path.join(cfg.out_dir, 'station_tags.csv'), dtype=str)
L = C.lau_with_city_units(load_lau_subset(cfg), cfg)          # Košice / Bratislava districts -> one unit each
tags = C.tags_with_city_units(tags, cfg, L)
hub_lau = C.hub_city_laus(cfg)                                 # the whole city of Košice
snap = pd.read_csv(os.path.join(cfg.out_dir, 'station_rail_snap.csv'), dtype={'station_key': str})
geo = tags.set_index('station_key')
lau_of = F.station_lau(tags, cfg)
pop, pop_year = F.lau_population(L)
lau_geo = tags.drop_duplicates('lau_gisco_id').set_index('lau_gisco_id')[['lau_name_national', 'nuts3_id', 'degurba']]
runs = F.country_runs(cfg)
print(f'rule: arrive by {cfg.daytrip_max_arrival}, dwell >= {cfg.daytrip_min_dwell_min} min, depart >= {cfg.daytrip_min_departure}, '
      f'back by {cfg.daytrip_max_return}; dwell sensitivity {cfg.daytrip_dwell_sensitivity}')

rule: arrive by 11:00, dwell >= 360 min, depart >= 17:00, back by 23:59; dwell sensitivity [240, 360, 480]


## 1. Options

In [3]:
legs = pd.read_parquet(os.path.join(OUT, 'direct_legs.parquet'))
opts = legs.rename(columns={'dep_s': 'start_s', 'arr_s': 'end_s'})[['run', 'direction', 'station_key', 'lau_gisco_id', 'start_s', 'end_s', 'mask']].assign(kind='direct')
if KIND == 'one_transfer':
    it = pd.read_parquet(os.path.join(OUT, 'one_transfer_itins.parquet'))
    opts = pd.concat([opts, it[['run', 'direction', 'station_key', 'lau_gisco_id', 'start_s', 'end_s', 'mask']].assign(kind='one_transfer')],
                     ignore_index=True)
print(opts.groupby(['run', 'direction', 'kind']).agg(options=('mask', 'size'), stations=('station_key', 'nunique')).unstack('direction'))

           options          stations         
direction  inbound outbound  inbound outbound
run kind                                     
CZ  direct      58       61       18       18
HU  direct      70       68       10       10
UA  direct       7        7        3        3


## 2. Day-trip days per station and per LAU

In [4]:
station, lau = [], []
for r in runs:
    N, W = len(r['days']), F.run_window(r)
    o = opts[opts.run == r['run']]
    inb, outb = o[o.direction == 'inbound'], o[o.direction == 'outbound']
    if inb.empty or outb.empty:
        continue
    station.append(C.daytrip_days(inb, outb, cfg, N, by='station_key').assign(country=r['country'], run=r['run'], **W))
    lau.append(C.daytrip_days(inb, outb, cfg, N, by='lau_gisco_id').assign(country=r['country'], run=r['run'], **W))
station = pd.concat(station, ignore_index=True)
station['lau_gisco_id'] = station['station_key'].map(lau_of)
station = (station.join(geo['stop_name'], on='station_key').join(lau_geo, on='lau_gisco_id')
           .join(snap.set_index('station_key')['rail_km'], on='station_key'))
station = station[['country', 'run', 'station_key', 'stop_name', 'lau_gisco_id', 'lau_name_national', 'nuts3_id', 'degurba',
                   'n_daytrip_days', 'daytrip_share', 'rail_km', 'window_start', 'window_end', 'window_days']]
lau = pd.concat(lau, ignore_index=True).join(lau_geo, on='lau_gisco_id')
lau['population'] = lau['lau_gisco_id'].map(pop)
lau['pop_year'] = lau['lau_gisco_id'].map(pop_year)
lau = lau[['country', 'run', 'lau_gisco_id', 'lau_name_national', 'nuts3_id', 'degurba', 'population', 'pop_year',
           'n_daytrip_days', 'daytrip_share', 'window_start', 'window_end', 'window_days']]
print(f"LAUs with >= 1 day-trip day: {lau.groupby('run').size().to_dict()}")
display(lau.sort_values('population', ascending=False).head(15))

LAUs with >= 1 day-trip day: {'HU': 10}


,country,run,lau_gisco_id,lau_name_national,nuts3_id,degurba,population,pop_year,n_daytrip_days,daytrip_share,window_start,window_end,window_days
9,HU,HU,HU_13578,Budapest,HU110,1,1675777.0,2025,275,0.913621,2026-01-19,2026-11-15,301
6,HU,HU,HU_30456,Miskolc,HU311,1,139246.0,2025,275,0.913621,2026-01-19,2026-11-15,301
8,HU,HU,HU_22309,Hatvan,HU312,2,19353.0,2025,275,0.913621,2026-01-19,2026-11-15,301
7,HU,HU,HU_19433,Mezőkövesd,HU311,2,15221.0,2025,275,0.913621,2026-01-19,2026-11-15,301
2,HU,HU,HU_33048,Encs,HU311,2,5962.0,2025,275,0.913621,2026-01-19,2026-11-15,301
5,HU,HU,HU_21351,Szikszó,HU311,2,5340.0,2025,275,0.913621,2026-01-19,2026-11-15,301
4,HU,HU,HU_27942,Halmaj,HU311,3,1734.0,2025,275,0.913621,2026-01-19,2026-11-15,301
1,HU,HU,HU_27191,Novajidrány,HU311,3,1279.0,2025,275,0.913621,2026-01-19,2026-11-15,301
3,HU,HU,HU_08086,Ináncs,HU311,3,1132.0,2025,275,0.913621,2026-01-19,2026-11-15,301
0,HU,HU,HU_11697,Hidasnémeti,HU311,3,950.0,2025,275,0.913621,2026-01-19,2026-11-15,301


## 3. Coverage, with its denominators

Per country: **all** LAUs; **served** (an option in both directions); a same-day visit on **at least one** day,
on **≥ 50 %** and on **≥ 90 %** of the run's days.

In [5]:
rows = []
for r in runs:
    c = r['country']
    allc = set(L.loc[L['CNTR_CODE'] == c, 'GISCO_ID'])
    o = opts[opts.run == r['run']]
    served = set(o.loc[o.direction == 'inbound', 'lau_gisco_id']) & set(o.loc[o.direction == 'outbound', 'lau_gisco_id'])
    sh = lau[lau.run == r['run']].set_index('lau_gisco_id')['daytrip_share']
    for g, ids in [('all_lau', allc), ('served', served), ('daytrip_any', set(sh.index)),
                   ('daytrip_50', set(sh[sh >= 0.5].index)), ('daytrip_90', set(sh[sh >= 0.9].index))]:
        ids = ids & allc
        rows.append({'country': c, 'run': r['run'], 'group': g, 'n_lau': len(ids), 'population': pop.reindex(list(ids)).sum(),
                     **F.run_window(r)})
cov = pd.DataFrame(rows)
base = cov[cov.group == 'all_lau'].set_index('run')
cov['share_lau'] = (cov['n_lau'] / cov['run'].map(base['n_lau'])).round(4)
cov['share_population'] = (cov['population'] / cov['run'].map(base['population'])).round(5)
cov = cov[['country', 'run', 'group', 'n_lau', 'population', 'share_lau', 'share_population', 'window_start', 'window_end', 'window_days']]
display(cov.pivot_table(index='run', columns='group', values='n_lau', aggfunc='sum'))
display(cov.pivot_table(index='run', columns='group', values='population', aggfunc='sum').map(lambda v: f'{v:,.0f}'))

group,all_lau,daytrip_50,daytrip_90,daytrip_any,served
run,,,,,
AT,2092,0,0,0,0
CZ,6258,0,0,0,14
HU,3155,10,10,10,10
PL,2477,0,0,0,0
PL:PL_IC,2477,0,0,0,0
UA,1769,0,0,0,3


group,all_lau,daytrip_50,daytrip_90,daytrip_any,served
run,,,,,
AT,"9,215,956",0,0,0,0
CZ,"10,915,839",0,0,0,"2,119,181"
HU,"9,488,428","1,865,994","1,865,994","1,865,994","1,865,994"
PL,"37,332,510",0,0,0,0
PL:PL_IC,"37,332,510",0,0,0,0
UA,"41,167,335",0,0,0,"240,896"


## 4. Dwell sensitivity (VAL-3)

In [6]:
sens = []
for r in runs:
    N = len(r['days'])
    o = opts[opts.run == r['run']]
    inb, outb = o[o.direction == 'inbound'], o[o.direction == 'outbound']
    for dwell in sorted(cfg.daytrip_dwell_sensitivity):
        d = C.daytrip_days(inb, outb, cfg, N, by='lau_gisco_id', dwell_min=dwell) if len(inb) and len(outb) else \
            pd.DataFrame(columns=['lau_gisco_id', 'daytrip_share'])
        h = d[d.daytrip_share >= 0.5]
        sens.append({'country': r['country'], 'run': r['run'], 'dwell_min': dwell, 'is_base': dwell == cfg.daytrip_min_dwell_min,
                     'n_lau_any': len(d), 'pop_any': pop.reindex(list(d['lau_gisco_id'])).sum(),
                     'n_lau_50': len(h), 'pop_50': pop.reindex(list(h['lau_gisco_id'])).sum(), **F.run_window(r)})
sens = pd.DataFrame(sens)
display(sens.drop(columns=['window_start', 'window_end']))

,country,run,dwell_min,is_base,n_lau_any,pop_any,n_lau_50,pop_50,window_days
0,CZ,CZ,240,False,0,0.0,0,0.0,364
1,CZ,CZ,360,True,0,0.0,0,0.0,364
2,CZ,CZ,480,False,0,0.0,0,0.0,364
3,HU,HU,240,False,10,1865994.0,10,1865994.0,301
4,HU,HU,360,True,10,1865994.0,10,1865994.0,301
5,HU,HU,480,False,0,0.0,0,0.0,301
6,UA,UA,240,False,0,0.0,0,0.0,364
7,UA,UA,360,True,0,0.0,0,0.0,364
8,UA,UA,480,False,0,0.0,0,0.0,364
9,PL,PL,240,False,0,0.0,0,0.0,32


## 5. Against the legacy run

The legacy run wrote `ke_daytrip_direct_station.csv` (its own station ids); matched by station name for the stations abroad.

In [7]:
old = pd.read_csv('output/ke/ke_daytrip_direct_station.csv', encoding='utf-8-sig').merge(
    pd.read_csv('output/ke/station_catalog.csv', encoding='utf-8-sig', dtype=str)[['station_id', 'name', 'country']], on='station_id')
old = old[(old['country'] != 'SK') & (old['n_daytrip_days'] > 0)][['name', 'country', 'n_daytrip_days']].rename(columns={'name': 'stop_name', 'n_daytrip_days': 'legacy_days'})
new = station[station.run.isin([r['run'] for r in runs if r['headline']])][['stop_name', 'country', 'n_daytrip_days', 'window_days']]
cmpd = new.merge(old, on='stop_name', how='outer', indicator=True)
print(cmpd['_merge'].value_counts().to_dict())
both = cmpd[cmpd._merge == 'both']
print(f"matched {len(both)} stations; the legacy run used the 364-day window for every country")
display(both.head(20))
display(cmpd[cmpd._merge != 'both'].head(20))

{'both': 10, 'left_only': 0, 'right_only': 0}
matched 10 stations; the legacy run used the 364-day window for every country


,stop_name,country_x,n_daytrip_days,window_days,country_y,legacy_days,_merge
0,Budapest-Keleti pu,HU,275,301,HU,338,both
1,Forro-Encs,HU,275,301,HU,338,both
2,Halmaj,HU,275,301,HU,338,both
3,Hatvan,HU,275,301,HU,338,both
4,Hidasnémeti,HU,275,301,HU,338,both
5,Inancs,HU,275,301,HU,338,both
6,Mezökövesd,HU,275,301,HU,338,both
7,Miskolc Tiszai pu,HU,275,301,HU,338,both
8,Novajidrany,HU,275,301,HU,338,both
9,Szikszo-Vasarter,HU,275,301,HU,338,both


,stop_name,country_x,n_daytrip_days,window_days,country_y,legacy_days,_merge


## 6. Outputs

In [8]:
for name, key, frame in [(f'daytrip_{KIND}_station.csv', f'foreign_daytrip_{KIND}_station', station),
                         (f'daytrip_{KIND}_lau.csv', f'foreign_daytrip_{KIND}_lau', lau),
                         (f'daytrip_{KIND}_coverage.csv', f'foreign_daytrip_{KIND}_coverage', cov),
                         (f'daytrip_{KIND}_sensitivity.csv', f'foreign_daytrip_{KIND}_sensitivity', sens)]:
    p = os.path.join(OUT, name)
    frame.to_csv(p, index=False, encoding='utf-8')
    write_meta(p, key, frame, {'Rule': f'arrive <= {cfg.daytrip_max_arrival}, dwell >= {cfg.daytrip_min_dwell_min} min, '
                                       f'depart >= {cfg.daytrip_min_departure}, back <= {cfg.daytrip_max_return}'})
    print(f'[wrote] {p} ({len(frame):,} rows) + .meta.md')

[wrote] output\kosice\foreign\daytrip_direct_station.csv (10 rows) + .meta.md
[wrote] output\kosice\foreign\daytrip_direct_lau.csv (10 rows) + .meta.md
[wrote] output\kosice\foreign\daytrip_direct_coverage.csv (30 rows) + .meta.md
[wrote] output\kosice\foreign\daytrip_direct_sensitivity.csv (18 rows) + .meta.md
